# AGENTIC RAG - RAGAS

This notebook walks through building an Agentic RAG system using LangChain and FAISS. It also demonstrates how to evaluate the RAG pipeline using RAGAS, a framework for evaluating Retrieval-Augmented Generation systems across key metrics like faithfulness, answer relevancy, context precision, and context recall.

## What is RAGAS and Why is it Essential?

RAGAS (Retrieval-Augmented Generation As A Service) is a powerful framework specifically designed to evaluate the performance of Retrieval-Augmented Generation (RAG) systems. It goes beyond traditional metrics like BLEU or ROUGE by focusing on aspects critical to the success of RAG applications.

**Why RAGAS is essential for RAG systems:**

1.  **Addresses RAG-specific challenges:** RAG systems combine information retrieval with text generation. This introduces unique failure modes, such as hallucination (generating information not present in the source), irrelevance to the query, or poor context utilization. RAGAS is built to directly measure these issues.

2.  **Focus on factual accuracy and grounding:** RAGAS's `Faithfulness` metric directly checks if the claims made in the generated answer are supported by the retrieved context. This is crucial for applications where factual correctness is paramount.

3.  **Evaluates relevance:** The `Answer Relevancy` metric assesses whether the generated answer directly addresses the user's question, ensuring the system doesn't produce off-topic responses.

4.  **Assesses retrieval quality:** Unlike general-purpose NLP metrics, RAGAS includes `Context Precision` and `Context Recall`. These metrics evaluate the effectiveness of the retrieval component by checking if the retrieved documents are relevant and comprehensive enough to answer the question.

5.  **Human-like evaluation:** RAGAS uses an LLM (the "judge LLM") to evaluate answers and contexts. This allows for a more nuanced assessment that can understand semantics and context, closer to how a human would evaluate the system, rather than just relying on lexical overlap.

6.  **Actionable insights:** The individual metrics provide clear signals about where a RAG system might be failing. For example, a low `Faithfulness` score suggests issues with the generation model (e.g., hallucination), while low `Context Recall` points to problems with the retrieval mechanism (e.g., missing relevant documents). This helps developers pinpoint and address specific weaknesses.

7.  **Automation:** RAGAS automates the evaluation process, which can be time-consuming and subjective if done manually, especially for large datasets. This enables continuous integration and testing for RAG pipelines.

In essence, RAGAS provides a holistic and specialized approach to RAG evaluation, giving developers the tools to build more reliable, accurate, and relevant RAG applications.

**Steps:**
1. Install dependencies
2. Ingest documents into a FAISS vector store
3. Build a LangChain agent with a document search tool
4. Ask questions

## Why Traditional ML and NLP Metrics Fall Short for RAG Evaluation

While traditional machine learning (ML) and natural language processing (NLP) evaluation metrics are valuable in their respective domains, they are often insufficient or misleading when it comes to comprehensively assessing Retrieval-Augmented Generation (RAG) systems. Here's why:

### 1. Classical ML Metrics (Precision, Accuracy, Recall, F1-score)

These metrics are primarily designed for **classification or structured prediction tasks**, where there's a clear, single correct output label or value. They measure how well a model predicts a predefined category or number.

*   **Mismatch with Generative Tasks:** RAG systems are generative; they produce free-form text answers. There isn't a fixed set of classes or a single 'correct' string output to compare against for metrics like accuracy or precision in the classical sense.
*   **Lack of Nuance for Text:** These metrics cannot capture the linguistic nuances, fluency, coherence, or semantic meaning of generated text. A RAG system might generate a perfectly accurate answer in terms of content, but if it's poorly phrased or irrelevant to the user's intent, classical metrics wouldn't reflect this.
*   **No Concept of 'Grounding':** Crucially, classical ML metrics have no mechanism to check if the generated answer is *grounded* in the retrieved context, which is a core requirement for RAG to avoid hallucination.

### 2. BLEU, ROUGE, and BERTScore (for NLP tasks)

These metrics are designed for evaluating text generation tasks like machine translation, summarization, or dialogue systems by comparing the generated text to one or more reference texts. While they address some aspects of text generation, they have significant limitations for RAG:

#### BLEU (Bilingual Evaluation Understudy) and ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

*   **Lexical Overlap Focus:** Both BLEU and ROUGE primarily rely on n-gram overlap between the generated answer and a reference answer. They measure lexical similarity rather than semantic understanding.
*   **Ignores Semantic Meaning:** A RAG system might generate an answer that uses different phrasing but conveys the exact same correct information as the reference. BLEU/ROUGE would penalize this for lack of exact word match, failing to capture semantic equivalence.
*   **Poor for Factuality/Hallucination:** These metrics do not inherently check if the generated text is factually correct or if it's supported by the retrieved source documents. A generated answer could perfectly match a reference (scoring high), but if both contain hallucinations, BLEU/ROUGE wouldn't detect it.
*   **Multiple Valid Answers:** For many questions in RAG, there can be multiple equally valid ways to phrase an answer. Relying on a single reference makes these metrics less robust, as they don't account for paraphrasing or alternative correct formulations.
*   **Context Disregard:** They don't evaluate *how* the retrieved context was used or if it was sufficient and relevant. They only compare the final generated text to a reference.

#### BERTScore

*   **Semantic Similarity:** BERTScore improves upon BLEU/ROUGE by using contextual embeddings (like BERT) to measure semantic similarity between tokens in the generated and reference texts. This makes it better at handling paraphrasing than n-gram based metrics.
*   **Still Lacks Grounding Check:** While semantically aware, BERTScore still doesn't inherently verify if the generated answer is factually grounded in the *retrieved documents*. It compares the generated answer to a *reference answer*, not directly to the source context that the RAG system *should* have used.
*   **Reference Dependency:** Like BLEU/ROUGE, it still requires high-quality reference answers, which can be expensive and time-consuming to produce, especially for complex RAG tasks where answers can be long and varied.

### Why RAGAS is Better Suited

RAGAS directly addresses these limitations by introducing metrics like `Faithfulness`, `Answer Relevancy`, `Context Precision`, and `Context Recall`. These metrics are specifically designed to evaluate the unique challenges of RAG systems by:

*   **Checking grounding against retrieved context:** `Faithfulness` and `Context Recall` directly assess the relationship between the generated answer, the retrieved context, and the original query.
*   **Evaluating relevance to the query:** `Answer Relevancy` ensures the answer actually addresses the question asked.
*   **Assessing retrieval effectiveness:** `Context Precision` and `Context Recall` directly measure the quality of the retrieved information, which is a critical component of RAG.
*   **Using LLM-as-a-judge:** This approach allows for more nuanced and semantic evaluation than purely lexical or statistical methods, better reflecting human judgment.

## 1. Install Dependencies

In [1]:
import os

if not os.path.exists('requirements.txt'):
    !wget https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-langgraph-02/requirements.txt
else:
    print('requirements.txt already exists.')
%pip install -r  requirements.txt -q


requirements.txt already exists.


## 2. Set API Keys

In [2]:
import os
from dotenv import load_dotenv

# Fallback: try Google Colab secrets
try:
    from google.colab import userdata
    if not os.getenv("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if not os.getenv("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY") or ""
except ImportError:
    # Load from .env file (local development)
    load_dotenv("/home/sirkumar/FDP-AGENENTIC-AI-RAG/.env")

HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip()
GEMINI_API_KEY = (os.getenv("GEMINI_API_KEY") or "").strip()


if not GEMINI_API_KEY or not HF_TOKEN:
    print("Warning: HF_TOKEN or GEMINI_API_KEY not set. Please set it in .env file or Colab secrets.")

## 3. Ingest Documents into FAISS Vector Store

Defines knowledge inline, splits it into chunks, embeds them, and saves the vector store locally.

In [3]:
department_info = [
    # Department overview
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.",
    "Students can choose from 5 courses ranging from core subjects to electives and hands-on project work, with strong industry exposure.",
    "DQE offers a postgraduate module 'Foundations of Quantum AI' and an undergraduate elective 'Quantum Machine Learning' using IBM Qiskit and PennyLane.",
    "All students have access to cloud quantum hardware through IBM Quantum Network and Amazon Braket as part of their coursework.",

    # Quantum AI research
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.",
    "A key research thread is Quantum Reinforcement Learning (QRL), where quantum agents learn policies faster than classical counterparts on specific problem classes.",
    "Project QuLearn is DQE's flagship initiative building hybrid classical-quantum models for drug discovery and materials science.",

]


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

documents = [Document(page_content=text) for text in department_info]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
docs = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2")
db = FAISS.from_documents(docs, embeddings)
db.save_local("vectorstore")

print(f"Vector DB created successfully ({len(docs)} chunks)")

/tmp/ipykernel_10123/3139761478.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector DB created successfully (8 chunks)


## 4. Build the RAG Agent

Loads the vector store and wires it up as a tool for a LangChain ZERO_SHOT_REACT agent.

In [5]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS

db = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True,
)

@tool
def search_answers(query: str) -> str:
    """Search the knowledge base"""
    results = db.similarity_search(query, k=2)
    return "\n".join([doc.page_content for doc in results])

# ── Switch providers by setting MODEL_NAME — no code changes needed ──────────
#   google_genai → MODEL_NAME=google_genai:gemini-2.5-flash   (default)
#   openai       → MODEL_NAME=openai:gpt-4o
#   anthropic    → MODEL_NAME=anthropic:claude-3-5-sonnet-20241022
#   huggingface  → MODEL_NAME=huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0
#   ollama       → MODEL_NAME=ollama:llama3
# ────────────────────────────────────────────────────────────────────────────

# Make loaded keys visible to init_chat_model via standard env-var names
#os.environ.setdefault("GOOGLE_API_KEY", GEMINI_API_KEY or "")

MODEL_NAME = "google_genai:gemini-2.5-flash"
MODEL_KEY= GEMINI_API_KEY
#MODEL_NAME = "huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0"
#MODEL_KEY = HF_TOKEN

# MODEL_NAME = "openai:gpt-4o"

chat_llm = init_chat_model(MODEL_NAME, temperature=0.1)

agent = create_agent(
    model=chat_llm,
    tools=[search_answers],
    system_prompt=(
        "You are an intelligent Agentic RAG system. "
        "Decide whether document retrieval is needed. "
        "If needed, use the search_answers tool."
    ),
)

print(f"Agent ready (model: {MODEL_NAME}).")

Agent ready (model: google_genai:gemini-2.5-flash).


## 5. Ask Questions

Edit `question` below and re-run this cell to query the agent.

In [6]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print("\nAnswer:", result["messages"][-1].content)


ask("What is DQE?")


Answer: [{'type': 'text', 'text': 'DQE stands for the Department of Quantum Engineering. Its primary research focus is Quantum AI, which is the intersection of quantum computing and artificial intelligence. The department has approximately 140 students and 20 professors, and they concentrate on applied quantum computing and intelligent systems.', 'extras': {'signature': 'CqkEAQw51sejju7lj6etS+v4HvlWXG703M/WcqPFeGnqXkIjoJZK6UjmruHZL+RfJXiqEhxf2mPyGh+QQud7KdwvDsxgCr4KPoA+hk/Qd1xNUQzHV433B7xl5Gfn9ws1uf6+NHavaI+e0gIRQemui73gKn1f68KcHvHKUw7w+U1sfpAwslkAWVR/HaatpY/FH/U7Owo/+iBMf7MbXwCHUNyqcffI49EIwSMlsM6xVQyootn7Xl4NzHjk7JbiheeLdTL5ttT08p0UXl7WNA03Ue9AKmAy0fMH/bZY6zKoP0H4A6CSA98BMapZLYz2NFeMAlSNUuM/yxzyqYv6GjhXLh0pDxhf7EpIdRuC5v+q731zYavIJdaQYr15gA/ASIDLoqdXywey+LvFnoNs+UtddBDi6pgKDFgom1IYX2ZVRUxgLxnEZPoU1xCknBKnbfueKCO8VfL5YmXPuAehLW4a1RU6Rviu5oqOefA6PhVQ+IGSercZtc8zj9slEx9EfMWtYtn59rZxkABSirapvnoUPwnoCUj6SoWLzzP4KAZwenujlNVDeuWxCG6+WoRwpPAp8md3Ff/0tZt7/Xj3A8meClzoskYZfKsiWz3JaPVNglJHrAOPx

In [7]:
ask("How many students does DQE have?")


Answer: [{'type': 'text', 'text': 'The Department of Quantum Engineering (DQE) has around 140 students.', 'extras': {'signature': 'Cu8CAQw51sfoEv4ocSm3BIBaNdCRmK+llqDPI3F5+C8gdRmCnM+n8Wp82//wbZ4evgqG4trE/MPvN1N7u/JeAf1RpwdZBfMIPHn7fvLgNblHnzJZ8hNZLwGM3eGIyFmvUhC6auHzZI4txx3XeOb+dQKpdRqy8uYrLZAQjX2O3oJhwGGpy2ZUQGmM1nZwIUkomPQfUJKNeP7syIBWbuto7c2tSKJzzN/Ncn1ea1CYnjXJG3/wCS5O40YgHS2BDYYJVZHOrkxf6te+tpbIqW3ZbHCInFPkWol+Xfx9Z5c9Wky7ucDQr61Moemq8xm9hCj6kat5UjzvAYLl6a2HecaYY/r2dWgTnYCvP0z/DS3WUYnK2KN8qJoKJo1Hx77ifGwz/uB0V55Z9qfZ7A3+8jg7vwL6BkTtnkF09O/6pvzMJW7yefXHCqiGPCwcb5Fe0RFGLwnTnplYL4y0iWQjZsypC3ewdhuNNvfG1QCOLKS+IIaIJw=='}}]


In [8]:
ask("How many stuents does it have")


Answer: [{'type': 'text', 'text': 'The Department of Quantum Engineering (DQE) has around 140 students.', 'extras': {'signature': 'CoEFAQw51scOct0mHr9cRr84CC3M2ypHg3GW+lk5+QwpPdgCQvwWR2kivlBoPlPbOBUVqq24l61ctowSeQWbp8yLLpGsC/P+Rqssxd+y2e8V8qye+2ZihaN4VWEMHk6693lbw2Qv0bS8fdCAmwFN9K0zPBecTT02RWa/RaoJ1byC/vlOHdE7uQObUjhuFGwK4ypdhg71wb6zLXG5WjMBCte2u2EMFNARfd4rrZPU8TkGStawRZPbqfRUXKk9P5hrw9Jc74eeleM7GzhBU2MzF8kvSHFbCNdtKbbU+fcSHAmpYhA+LRxmGktQCJBzjkcYR98JG1tIIBGfPYkyHSPF39pY9/gafUUrBeGEIlNsMhFjtLIDJA8Hgf3YwD0yageOqepBQrmNncl5/28C+iRww40NCmkcUWySNM9YgaKDkNV33IG194pOtSZMgartusdaqcwz1s3lgmWLkg3/LCZRVl1XwyJXO2hpPcvMRufl2CeoQ5vnwulLHcXTm7FNRU+UERg5Zy4CzH01ExuDiuXrZvL4BUAsM6ogiDjYaffTFkobYykVAkGA+4bxfs35l14GdooDJGqfNYG87WpcWkro0YokxYv23bGChdSssCqK8zpqJA87inMNjEhic9lw8PUoChewwLjnhgtxwH05In76iDJl60i/6bS1GAQ/YAZjAYvSbleKv9bAYn6psBQAar6713i5ih7Ol45qJrqStu6ruaKaKVs2OEmJV6I1FKtjgOTmZMtJbvjEsER46HIiltzbf3at5TpaXLPbwD8SW9F19SaeGN0fPy7d5syvonA4nw5tajqCPtzOLMqEZ/3jzh9a2kmUKjdzYwbNtXMdqSLIZhjCcoY='}}]


## 6. Evaluate with RAGAS

[RAGAS](https://docs.ragas.io/) measures RAG pipeline quality across four metrics:

| Metric | What it measures | Needs ground truth? |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context? | No |
| **Answer Relevancy** | Is the answer relevant to the question? | No |
| **Context Precision** | Are the retrieved chunks actually useful? | Yes |
| **Context Recall** | Did retrieval capture what was needed to answer? | Yes |

Scores range from 0 to 1 — higher is better.

## 7. RAGAS

### Retrieval-Augmented Generation As A Service. It's a framework for evaluating the performance of Retrieval-Augmented Generation (RAG) systems.

### What each metric tells you

| Metric | Question it answers | Score guide |
|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved chunks? | < 0.5 → hallucination risk; > 0.8 → trustworthy |
| **Answer Relevancy** | Does the answer actually address the question asked? | < 0.5 → off-topic; > 0.8 → on-point |
| **Context Precision** | Are the top-ranked retrieved chunks the useful ones? | < 0.5 → noisy retrieval; > 0.8 → precise retrieval |
| **Context Recall** | Did retrieval surface everything needed to answer? | < 0.5 → missing evidence; > 0.8 → good coverage |

---

### Reading the scores in this pipeline

**Faithfulness** measures whether the model "sticks to the script." A small model like TinyLlama-1.1B tends to score lower here because it may add plausible-sounding details not present in the retrieved text (hallucination). A cloud model like Gemini typically scores higher because it follows instructions more precisely.

**Answer Relevancy** reflects how well the question is understood. Short, focused answers on a narrow knowledge base (like DISE department info) usually score well. If the agent wanders off-topic or refuses to answer, this drops.

**Context Precision & Recall** are about retrieval quality, not the LLM. Since we use a small FAISS store (5 chunks) with sentence-transformer embeddings, precision is expected to be high — there is little noise to retrieve. Recall may suffer on multi-fact questions if the relevant chunk wasn't ranked in the top-k.

---

### What to do if scores are low

| Symptom | Likely cause | Fix |
|---|---|---|
| Low Faithfulness | LLM adds facts not in context | Use a stronger/instruction-tuned model; tighten the system prompt |
| Low Answer Relevancy | Model misunderstands the question | Improve the system prompt; switch to a better model |
| Low Context Precision | Wrong chunks retrieved | Tune `k`, try MMR retrieval, or re-embed with a better model |
| Low Context Recall | Needed chunk not retrieved | Increase `k`, reduce chunk size, or improve the embedding model |

---

> **Note on the judge LLM:** RAGAS uses the same `judge_llm` to score faithfulness and relevancy. If the judge is also TinyLlama (a small model), scores may be noisy or inconsistent — a stronger model (Gemini, GPT-4o) as the judge gives more reliable scores even when TinyLlama is the RAG agent.

In [9]:
def extract_answer(agent_response: dict) -> str:
    """Pull the plain-text answer out of the agent's message (handles list or str content)."""
    content = agent_response["messages"][-1].content
    if isinstance(content, list):
        return " ".join(
            item.get("text", "") for item in content if isinstance(item, dict)
        ).strip()
    return str(content).strip()


# --- Ground-truth answers (reference) ---
eval_questions = [
    "What is DQE?",
    "How many students does DQE have?",
    "How many professors are in DQE?",
    "What courses are offered in DQE?",
    "What is the primary research focus of DQE?"
]

ground_truths = [
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems. DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "DQE has around 140 students.",
    "DQE has around 20 professors.",
    "DQE offers 5 courses, including core subjects, electives, hands-on project work, a postgraduate module 'Foundations of Quantum AI', and an undergraduate elective 'Quantum Machine Learning'.",
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence. They investigate Quantum Neural Networks (QNNs) and Quantum Reinforcement Learning (QRL), and their flagship initiative is Project QuLearn."
]

# --- Collect RAG answers and retrieved contexts ---
answers, contexts = [], []

for question in eval_questions:
    # Retrieve chunks used as context
    retrieved = db.similarity_search(question, k=2)
    contexts.append([doc.page_content for doc in retrieved])

    # Get the agent's answer
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    answers.append(extract_answer(response))

# Preview
for q, a, ctx in zip(eval_questions, answers, contexts):
    print(f"Q: {q}")
    print(f"A: {a}")
    print(f"Ctx[0]: {ctx[0][:80]}...")
    print()

Q: What is DQE?
A: DQE stands for the Department of Quantum Engineering. Its primary research focus is Quantum AI, which is the intersection of quantum computing and artificial intelligence. The department has approximately 140 students and 20 professors, and they concentrate on applied quantum computing and intelligent systems.
Ctx[0]: DQE's primary research focus is Quantum AI — the intersection of quantum computi...

Q: How many students does DQE have?
A: The Department of Quantum Engineering (DQE) has around 140 students.
Ctx[0]: The Department of Quantum Engineering (DQE) has around 140 students and 20 profe...

Q: How many professors are in DQE?
A: DQE has 20 professors.
Ctx[0]: The Department of Quantum Engineering (DQE) has around 140 students and 20 profe...

Q: What courses are offered in DQE?
A: The Department of Quantum Engineering (DQE) offers 5 courses, including core subjects, electives, and hands-on project work, with strong industry exposure. However, the specific name

In [10]:
import sys
from types import ModuleType

# ragas tries to import ChatVertexAI from the stub — add a dummy class to satisfy it
for _mod_name, _attrs in {
    "langchain_community.chat_models.vertexai": ["ChatVertexAI"],
    "langchain_community.chat_models.google_palm": ["ChatGooglePalm"],
    "langchain_community.llms.vertexai": ["VertexAI"],
}.items():
    _mod = sys.modules.setdefault(_mod_name, ModuleType(_mod_name))
    for _attr in _attrs:
        if not hasattr(_mod, _attr):
            setattr(_mod, _attr, type(_attr, (), {}))

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_llm = LangchainLLMWrapper(chat_llm)
judge_embeddings = LangchainEmbeddingsWrapper(embeddings)

samples = [
    SingleTurnSample(
        user_input=q,
        response=a,
        retrieved_contexts=ctx,
        reference=gt,
    )
    for q, a, ctx, gt in zip(eval_questions, answers, contexts, ground_truths)
]
ragas_dataset = EvaluationDataset(samples=samples)

result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        Faithfulness(llm=judge_llm),
        AnswerRelevancy(llm=judge_llm, embeddings=judge_embeddings),
        ContextPrecision(llm=judge_llm),
        ContextRecall(llm=judge_llm),
    ],
)

import pandas as pd

scores_df = result.to_pandas()[
    ["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]
]
scores_df.loc["mean"] = scores_df.mean(numeric_only=True)
scores_df.iloc[-1, 0] = "MEAN"

pd.set_option("display.float_format", "{:.3f}".format)
print(scores_df.to_string(index=False))

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_10123/3602951817.py:16: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_10123/3602951817.py:16: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

                                user_input  faithfulness  answer_relevancy  context_precision  context_recall
                              What is DQE?         1.000             0.688              1.000           1.000
          How many students does DQE have?         1.000             0.817              1.000           1.000
           How many professors are in DQE?         1.000             0.979              1.000           1.000
          What courses are offered in DQE?         1.000             0.000              0.000           0.000
What is the primary research focus of DQE?         1.000             0.981              1.000           0.500
                                      MEAN         1.000             0.693              0.800           0.700
